# Convolution Neural Networks

## Designing a CNN model

In [13]:
import torch
import torchvision
from torchvision import datasets
from torch import nn
import torch.nn.functional as F


Implementation of Simple Convolution Neural Network. This is actually the AlexNet architecture. But the classifier is different as here we are using only two classes. 

In [14]:
class CNNNet(nn.Module):
    def __init__(self,num_classes=2):
        super(CNNNet,self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,64,kernel_size=11, stride=4, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6,6))
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256*6*6,4096),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(4096,4096),
            nn.ReLU(),
            nn.Linear(4096,num_classes)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x,1)
        x=self.classifier(x)

        return x

Let's try out our CNN for Images Classification task.  
We will use the model traininf function implemented in the previous notebook(ImageClassifier).

In [15]:
import torchvision
from torchvision import transforms
from torch.utils import data

transforms= transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229, 0.224, 0.225])
])

dataset = torchvision.datasets.CIFAR10(root='kaggle/input', 
                                        train=True, 
                                        download=True,
                                         transform=transforms)

test_dataset = torchvision.datasets.CIFAR10(root='kaggle/input', 
                                        train=False, 
                                        download=True,
                                        transform=transforms)



# Define the sizes for the split (80% train, 20% validation)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

# Set a seed for reproducibility
torch.manual_seed(42)

# Perform the split
train_dataset, val_dataset = data.random_split(dataset, [train_size, val_size])

# Print the sizes of each dataset to verify
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")

Training set size: 40000
Validation set size: 10000
Test set size: 10000


In [21]:
batch_size=256
train_data_loader= data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_data_loader= data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_data_loader= data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [22]:
import torch.optim as optim
from tqdm import tqdm



def train_model(model,optimizer,loss_fn,train_loader,val_loader,epochs=20,device='cpu'):
    best_val_loss = float('inf')
    best_model=None
    
    for epoch in range(epochs):
        train_loss = 0.0
        val_loss = 0.0
        model.train()
        for batch in tqdm(train_loader):
            optimizer.zero_grad()
            inputs,target=batch
            inputs=inputs.to(device)
            target=target.to(device)
            output = model(inputs)
            loss = loss_fn(output,target)
            loss.backward()
            optimizer.step()
            train_loss+=loss.data.item()*inputs.size(0)
        train_loss/=len(train_loader.dataset)

        model.eval()
        num_correct=0
        num_examples=0
        with torch.no_grad():
            for batch in tqdm(val_loader):
                inputs,target=batch
                inputs=inputs.to(device)
                target=target.to(device)
                output = model (inputs)
                loss = loss_fn (output,target)
                val_loss+=loss.data.item()*inputs.size(0)
                predicted = torch.argmax(output, dim=1)
                num_correct += (predicted == target).sum().item()
                num_examples += len(target)
        val_loss/=len(val_loader.dataset)
        accuracy = num_correct/num_examples


        print('Epoch: {} , Training loss: {:.2f}, Validation loss: {:.2f}, accuracy: {:.2f}'.format(epoch,train_loss,val_loss,accuracy))




In [23]:
model = CNNNet(num_classes=10)
model.to('cuda')

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
epochs = 20

In [24]:
train_model(model,optimizer,loss_fn,train_data_loader,val_data_loader,epochs=epochs,device='cuda')

100%|██████████| 40/40 [00:15<00:00,  2.51it/s]


Epoch: 0 , Training loss: 1.75, Validation loss: 1.48, accuracy: 0.47


100%|██████████| 40/40 [00:15<00:00,  2.54it/s]


Epoch: 1 , Training loss: 1.35, Validation loss: 1.23, accuracy: 0.56


100%|██████████| 40/40 [00:15<00:00,  2.52it/s]


...

Epoch: 17 , Training loss: 0.18, Validation loss: 0.63, accuracy: 0.81


100%|██████████| 40/40 [00:15<00:00,  2.53it/s]


Epoch: 18 , Training loss: 0.16, Validation loss: 0.62, accuracy: 0.81


100%|██████████| 40/40 [00:15<00:00,  2.52it/s]

Epoch: 19 , Training loss: 0.14, Validation loss: 0.64, accuracy: 0.82


In [26]:
torch.save(model.state_dict(),"/kaggle/working/CNN")   #saving the model params

Now Let's take a glance at the design of popular architectures 

In [ ]:
import torchvision.models as models 
model = models.mobilenet_v2()

print(model)

In [ ]:
print(models.googlenet(init_weights = True))

In [ ]:
print(models.vgg16())

In [ ]:
import torch
torch.hub.list('pytorch/vision')